In [ ]:
import subprocess, tempfile
from pathlib import Path
from PIL import Image
from IPython.display import display

REPO       = Path("/home/math/gupta/work/greek_byzantine/greek-foundation")
TEST_DIR   = REPO / "dataset" / "calamari" / "test"
CHECKPOINT = REPO / "outputs" / "calamari-greek-bible" / "best.ckpt"

test_images = sorted(TEST_DIR.glob("*.png"))
print(f"{len(test_images)} test images")

# Predict into a temp directory (auto-cleaned, you never see it)
tmp = tempfile.mkdtemp()
cmd = [
    str(REPO / ".venv/bin/calamari-predict"),
    "--checkpoint", str(CHECKPOINT),
    "--data.images", *[str(p) for p in test_images],
    "--data.pred_extension", ".pred.txt",
    "--output_dir", tmp,
    "--predictor.device.gpus", "0",
]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stderr[-1000:])
    raise RuntimeError("calamari-predict failed")

# Read predictions into a dict {stem: predicted_text}
predictions = {}
for f in Path(tmp).glob("*.pred.txt"):
    predictions[f.stem] = f.read_text("utf-8").strip()

print(f"Done. {len(predictions)} predictions loaded.")

In [ ]:
# Show: image | ground truth | prediction
N = 20  # number of samples to show

for img_path in test_images[:N]:
    gt   = (TEST_DIR / f"{img_path.stem}.gt.txt").read_text("utf-8").strip()
    pred = predictions.get(img_path.stem, "\u27e8no prediction\u27e9")

    print(f"\n{'\u2500'*80}")
    display(Image.open(img_path))
    print(f"  TRUE: {gt}")
    print(f"  PRED: {pred}")
    print(f"  {'\u2713 MATCH' if gt == pred else '\u2717 MISMATCH'}")